# Week 4: Folium and First Heatmap

> **Milestone:** Folium: First interactive heatmap
> **Deliverable:** Notebook 04 complete

*From PROJECT_BRIEF.md - Phase 1: Foundation (Weeks 1-4)*

In [ ]:
# Week 4: Create Folium interactive heatmap
import geopandas as gpd
import folium
from folium.plugins import MousePosition
import os

wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)

# Load data
basins = gpd.read_file('data/raw/usgs_basins.shp')
users = gpd.read_file('data/raw/end_users.shp')
basins = basins.set_crs(epsg=4326)
users = users.set_crs(epsg=4326)

# Create map centered on study area
m = folium.Map(location=[-15.0, 17.0], zoom_start=6, tiles='OpenStreetMap')

# Add basins as GeoJSON with tooltips
folium.GeoJson(
    basins,
    name='Geology Basins',
    style_function=lambda x: {
        'fillColor': 'yellow' if x['properties'].get('porosity', 0) > 0.25 else 'red',
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.5,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['basin_id', 'rock_type', 'porosity'],
        aliases=['ID:', 'Rock Type:', 'Porosity:'],
        localize=True,
    ),
).add_to(m)

# Add users as markers - use centroid coordinates
for _, user in users.iterrows():
    geom = user.geometry
    try:
        lat = geom.y
        lon = geom.x
    except:
        centroid = geom.centroid
        lat = centroid.y
        lon = centroid.x
    
    folium.Marker(
        location=[lat, lon],
        popup=f"{user.get('facility_type', 'User')}<br>Capacity: {user.get('capacity_mw', 'N/A')} MW",
        icon=folium.Icon(color='blue', icon='info-sign'),
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save
m.save('outputs/folium_heatmap.html')

print('✅ Folium heatmap saved!')
print('Files in outputs:', os.listdir('outputs'))